In [4]:
import os
import re
from datetime import datetime

import ee
import geemap  # (not strictly needed below, but fine to keep)


#####################################
#    SET ALL PARAMETERS #############
#####################################

PROJECT_DIR = "projects/ee-tymc5571-cbi-module/assets"
GEE_PROJECT = "ee-tymc5571-cbi-module"
DRIVE_FOLDER      = "GEE_Exports"
BCCBI_ASSET_STRING = "west_wall_to_wall_bccbi"
CRS = "EPSG:5070"  # export CRS
FILE_MANIFESTS = ["wumi_west_nocircles_manifest.csv",
                  "wumi_west_circles_manifest.csv",
                  "firedv2_west_manifest.csv",
                  "welty_west_manifest.csv",
                  ] #Export manifests from script 02

# ee.Authenticate()
ee.Authenticate(
    auth_mode='notebook',
    scopes=[
        'https://www.googleapis.com/auth/earthengine',
        'https://www.googleapis.com/auth/devstorage.read_write',
        'https://www.googleapis.com/auth/drive'
    ]
)
ee.Initialize(project=GEE_PROJECT)

SERVICE_FILE = "C:/Users/tymc5571/dev/cbi-module/config/secrets/tymc5571-utils-project-692f27c034dd.json"  # Path to service account file
DIR_DERIVED = "C:/Users/tymc5571/dev/cbi-module/data/derived/"


In [5]:
import os
import shutil
import tempfile
import contextlib
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Optional, List
import time, random

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from googleapiclient.http import MediaIoBaseDownload

# GDAL bindings
from osgeo import gdal


def list_all_files_in_folder(
    service,
    folder_id: str,
    description: Optional[str] = None,
    *,
    include_all_drives: bool = True,
    drive_id: Optional[str] = None,
    page_size: int = 1000,
) -> List[dict]:
    """
    List ALL files in a Drive folder, with pagination.
    Filters by description substring and .tif/.TIF extension.
    Returns a list of dicts with at least {'id','name'}.
    """
    # Query: in folder, not trashed, optional name contains 'description',
    # and extension contains '.tif' or '.TIF'
    name_filter = []
    if description:
        name_filter.append(f"(name contains '{description}')")
    ext_filter = "((name contains '.tif') or (name contains '.TIF'))"
    q_parts = [f"'{folder_id}' in parents", "trashed=false", ext_filter]
    if name_filter:
        q_parts.append(" and ".join(name_filter))
    q = " and ".join(q_parts)

    kwargs = dict(
        q=q,
        pageSize=page_size,
        fields="nextPageToken, files(id, name)",
        orderBy="name",
    )
    if include_all_drives:
        kwargs.update(dict(supportsAllDrives=True, includeItemsFromAllDrives=True))
        if drive_id:
            kwargs.update(dict(corpora="drive", driveId=drive_id))
        else:
            kwargs.update(dict(corpora="allDrives"))
    else:
        kwargs.update(dict(corpora="user"))

    files, page_token = [], None
    while True:
        resp = service.files().list(pageToken=page_token, **kwargs).execute()
        files.extend(resp.get("files", []))
        page_token = resp.get("nextPageToken")
        if not page_token:
            break
    return files

def _download_file_from_drive(file_id, file_name, temp_dir, service_account_file):
    try:
        # Authenticate service account
        SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
        credentials = service_account.Credentials.from_service_account_file(
            os.path.abspath(service_account_file), scopes=SCOPES
        )
        service = build('drive', 'v3', credentials=credentials)

        request = service.files().get_media(fileId=file_id)
        local_path = os.path.join(temp_dir, file_name)
        with open(local_path, 'wb') as f:
            downloader = MediaIoBaseDownload(f, request)
            done = False
            while not done:
                status, done = downloader.next_chunk()
        time.sleep(random.uniform(0.5, 1.5))  # Random delay
        print(f"Downloaded {file_name}")
        return local_path
    except Exception as e:
        print(f"Error downloading {file_name}: {e}")
        return None


def download_merge_from_drive_tif(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output GeoTIFF
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise RuntimeError("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to GeoTIFF (no giant array in RAM) ---
        co = [
            f"BIGTIFF={bigtiff}",
            "TILED=YES",
            f"COMPRESS={compress.upper()}",
            f"BLOCKXSIZE={int(blocksize)}",
            f"BLOCKYSIZE={int(blocksize)}",
            "OVERVIEWS=AUTO", #pyramids
            "RESAMPLING=AVERAGE",
            "NUM_THREADS=ALL_CPUS"
        ]
        if predictor is not None:
            co.append(f"PREDICTOR={int(predictor)}")

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))
        if needs_warp:
            if verbose:
                print("Warping (streamed) to target grid...")
            warp_kwargs = dict(
                format="GTiff",
                creationOptions=co,
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_ds = gdal.Warp(local_filename, vrt_path, **warp_kwargs)
            if out_ds is None:
                raise RuntimeError("GDAL Warp failed.")
            out_ds = None
        else:
            if verbose:
                print("Translating VRT to tiled, compressed GeoTIFF (streamed)...")
            out_ds = gdal.Translate(local_filename, vrt_path, creationOptions=co)
            if out_ds is None:
                raise RuntimeError("GDAL Translate failed.")
            out_ds = None

        if verbose:
            print(f"Final mosaicked GeoTIFF saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")


def download_merge_from_drive_cog(
    description: str,
    local_filename: str,
    drive_folder: str,
    service_account_file: str,
    *,
    # Output & mosaic controls
    compress: str = "DEFLATE",          # DEFLATE|LZW|ZSTD|JPEG
    predictor: Optional[int] = None,     # 2 for integers, 3 for floats; None = let GDAL decide
    blocksize: int = 512,                # tile size for output COG (both x and y)
    bigtiff: str = "YES",                # YES|IF_SAFER
    dst_srs: Optional[str] = None,       # e.g., "EPSG:5070" (if reprojection needed)
    xres: Optional[float] = None,        # set both xres & yres to enforce pixel size (e.g., 30)
    yres: Optional[float] = None,
    dst_nodata: Optional[float] = None,  # set if you need a specific nodata in output
    # Drive / listing controls
    drive_id: Optional[str] = None,      # if folder is in a Shared Drive, you can pass the Drive ID
    include_all_drives: bool = True,     # search across Shared Drives
    # Behavior controls
    check_existing: bool = True,
    n_workers: int = 4,
    verbose: bool = True,
) -> str:
    """
    Download all Drive .tif/.TIF files whose names contain `description` from `drive_folder`,
    then build a VRT and stream a mosaicked GeoTIFF to `local_filename` (no in-memory merge).
    """

    # --- Fast exit if we already have the target ---
    local_filename = os.path.abspath(local_filename)
    if check_existing and os.path.exists(local_filename):
        if verbose:
            print(f"File already exists: {local_filename}")
        return local_filename

    # --- Temp workspace ---
    temp_dir = tempfile.mkdtemp()
    vrt_path = os.path.join(temp_dir, "mosaic.vrt")
    downloaded_paths: List[str] = []

    # --- Build Drive client ---
    SCOPES = ["https://www.googleapis.com/auth/drive.readonly"]
    cred_path = os.path.abspath(service_account_file)
    credentials = service_account.Credentials.from_service_account_file(cred_path, scopes=SCOPES)
    service = build("drive", "v3", credentials=credentials)

    # Convenience flags for Shared Drives
    list_kwargs = dict(
        fields="nextPageToken, files(id, name, parents)",
        supportsAllDrives=True,
        includeItemsFromAllDrives=True
    ) if include_all_drives else dict(fields="nextPageToken, files(id, name, parents)")

    try:
        # --- Locate the folder (paginate just in case) ---
        folder_q = (
            f"name='{drive_folder}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        )
        folder_kwargs = dict(
            q=folder_q,
            pageSize=1000,
            corpora=("drive" if drive_id else "allDrives") if include_all_drives else "user",
            driveId=drive_id,
            **list_kwargs
        )

        folders, page_token = [], None
        while True:
            resp = service.files().list(pageToken=page_token, **folder_kwargs).execute()
            folders.extend(resp.get("files", []))
            page_token = resp.get("nextPageToken")
            if not page_token:
                break

        if not folders:
            raise FileNotFoundError(
                f"Folder '{drive_folder}' not found. "
                "Ensure it exists and is shared with the service account "
                f"({credentials.service_account_email}). "
                "If it is on a Shared Drive, also ensure the service account has at least Viewer."
            )
        folder_id = folders[0]["id"]

        # --- List ALL matching .tif files in that folder (paginated) ---
        files = list_all_files_in_folder(
            service,
            folder_id,
            description=description,
            include_all_drives=include_all_drives,
            drive_id=drive_id,
            page_size=1000
        )

        if not files:
            raise FileNotFoundError(
                f"No matching .tif/.TIF files found for '{description}' in '{drive_folder}'. "
                "Check filenames and sharing."
            )

        if verbose:
            print(f"Found {len(files)} file(s). Starting download to {temp_dir} ...")

        # --- Download the files (parallel if requested) ---
        if n_workers <= 1:
            for f in files:
                path = _download_file_from_drive(f["id"], f["name"], temp_dir, service_account_file)
                if path:
                    downloaded_paths.append(path)
        else:
            with ThreadPoolExecutor(max_workers=n_workers) as ex:
                futures = [
                    ex.submit(_download_file_from_drive, f["id"], f["name"], temp_dir, service_account_file)
                    for f in files
                ]
                for fut in as_completed(futures):
                    p = fut.result()
                    if p:
                        downloaded_paths.append(p)

        if not downloaded_paths:
            raise RuntimeError("Downloads finished but no files were saved.")

        if verbose:
            print(f"Building VRT from {len(downloaded_paths)} tile(s) ...")

        # --- Build VRT (lightweight, no data copy) ---
        vrt_options = {}
        if dst_nodata is not None:
            vrt_options.update(dict(srcNodata=dst_nodata, VRTNodata=dst_nodata))

        vrt = gdal.BuildVRT(vrt_path, downloaded_paths, **vrt_options)
        if vrt is None:
            raise RuntimeError("GDAL BuildVRT failed.")
        vrt = None  # flush

        # --- Stream out to Cloud-Optimized GeoTIFF (COG) ---
        # COG creation options (driver: "COG")
        co_cog = [
            f"COMPRESS={compress.upper()}",     # e.g., DEFLATE|LZW|ZSTD|JPEG
            f"BLOCKSIZE={int(blocksize)}",      # one value for both X/Y, typical 512
            f"NUM_THREADS={'ALL_CPUS'}",
            f"BIGTIFF={bigtiff}",               # YES|IF_SAFER
            "OVERVIEW_RESAMPLING=AVERAGE",      # build internal overviews with this kernel
            # Optional: "SPARSE_OK=YES",       # can help on sparse rasters
        ]
        if predictor is not None:
            co_cog.append(f"PREDICTOR={int(predictor)}")  # 2=int, 3=float (for DEFLATE/LZW)

        needs_warp = any(v is not None for v in (dst_srs, xres, yres, dst_nodata))

        if needs_warp:
            if verbose:
                print("Warping to target grid (VRT) and writing COG...")
            # Warp to a VRT (no data copy), then translate that VRT → COG
            warped_vrt = os.path.join(temp_dir, "warped.vrt")
            warp_kwargs = dict(
                format="VRT",           # keep it as VRT to avoid creating a big intermediate
                multithread=True,
            )
            if dst_srs:
                warp_kwargs["dstSRS"] = dst_srs
            if xres and yres:
                warp_kwargs["xRes"] = float(xres)
                warp_kwargs["yRes"] = float(yres)
                warp_kwargs["targetAlignedPixels"] = True
            if dst_nodata is not None:
                warp_kwargs["dstNodata"] = dst_nodata

            out_warp = gdal.Warp(warped_vrt, vrt_path, **warp_kwargs)
            if out_warp is None:
                raise RuntimeError("GDAL Warp (to VRT) failed.")
            out_warp = None

            # Final: VRT → COG
            out_ds = gdal.Translate(
                local_filename, warped_vrt,
                format="COG",
                creationOptions=co_cog
            )
            if out_ds is None:
                raise RuntimeError("GDAL Translate to COG failed.")
            out_ds = None

        else:
            if verbose:
                print("Translating VRT directly to COG (internal overviews)...")
            out_ds = gdal.Translate(
                local_filename, vrt_path,
                format="COG",
                creationOptions=co_cog
            )
            if out_ds is None:
                raise RuntimeError("GDAL Translate to COG failed.")
            out_ds = None

        if verbose:
            print(f"Final COG saved to: {local_filename}")
        return local_filename

    except HttpError as e:
        msg = str(e)
        if "accessNotConfigured" in msg or "has not been used in project" in msg:
            raise RuntimeError(
                "Google Drive API is disabled for this service account’s project.\n"
                "Enable the **Google Drive API** in GCP → APIs & Services → Library, then retry."
            ) from e
        raise
    except KeyboardInterrupt:
        print("Interrupted by user. Cleaning up and exiting.")
        raise
    finally:
        # Best-effort cleanup
        try:
            shutil.rmtree(temp_dir)
            if verbose:
                print(f"Cleaned up temporary directory: {temp_dir}")
        except Exception as cleanup_err:
            print(f"Error during cleanup: {cleanup_err}")

In [ ]:
import pandas as pd

for manifest in FILE_MANIFESTS:
    manifest_data = pd.read_csv(os.path.join(DIR_DERIVED, manifest))
    for filename in manifest_data['id']:
        local_path = os.path.join(DIR_DERIVED, f"{filename}.tif")

        # if local path exists, skip
        if os.path.exists(local_path):
            print(f"File already exists, skipping: {local_path}")
            continue

        print(f"COG processing {filename} from manifest {manifest}...")
        download_merge_from_drive_cog(
            description=filename,
            local_filename=local_path,
            drive_folder="GEE_Exports",
            service_account_file=SERVICE_FILE,
            compress="ZSTD",         
            predictor=3,                # 3 = best for float32 data (like indices or continuous values)
            blocksize=512,              # tile size; 256–512 is good default
            bigtiff="YES",              # ensures large mosaics don’t overflow 4 GB
            #dst_srs="EPSG:5070",        # optional: force reprojection to your CONUS Albers grid
            #xres=30, yres=30,           # optional: force 30 m pixels
            #dst_nodata=-9999,           # optional: set output NoData if needed
            check_existing=True,
            n_workers=8,                # parallel downloads from Drive
            include_all_drives=True,    # safer if GEE_Exports is on a Shared Drive
            verbose=True
        )

# download_merge_from_drive_cog(
#     description="bccbi_welty_west_2020",
#     local_filename=os.path.join(DIR_DERIVED, "test_cog.tif"),
#     drive_folder="GEE_Exports",
#     service_account_file=SERVICE_FILE,
#     compress="DEFLATE",         # same as 'deflate', case-insensitive
#     predictor=3,                # 3 = best for float32 data (like indices or continuous values)
#     blocksize=512,              # tile size; 256–512 is good default
#     bigtiff="YES",              # ensures large mosaics don’t overflow 4 GB
#     #dst_srs="EPSG:5070",        # optional: force reprojection to your CONUS Albers grid
#     #xres=30, yres=30,           # optional: force 30 m pixels
#     #dst_nodata=-9999,           # optional: set output NoData if needed
#     check_existing=True,
#     n_workers=4,                # parallel downloads from Drive
#     include_all_drives=True,    # safer if GEE_Exports is on a Shared Drive
#     verbose=True
# )

COG processing bccbi_wumi_west_nocircles_1985 from manifest wumi_west_nocircles_manifest.csv...
Found 6 file(s). Starting download to C:\Users\tymc5571\AppData\Local\Temp\tmpds8x8yvd ...
Downloaded bccbi_wumi_west_nocircles_1985-0000065536-0000032768.tif
